<!-- [intro-a] -->
# MCP Bridge — Internals

Why `mcp_tools_as_langchain_tools` (`mcp_tools.py`) behaves the way it does: closures, schema
passthrough, and why re-entering an already-used `client` doesn't error. Built from a real
debugging session against this exact function, not written ahead of investigating it. Pairs with
`01-architecture-overview.ipynb`, section C — that notebook has the round-trip picture; this one
has the mechanics behind each step.

In [1]:
# [setup-a]
# Scaffolding only — a tiny calc server so every cell below has real tools to work with,
# without paying for the real PDF ingestion pipeline. Nothing to study here, just run it once.
import asyncio

from fastmcp import Client, FastMCP
from langchain_core.tools import StructuredTool

mcp = FastMCP(name="CalcServer", instructions="Provides basic arithmetic tools.")


@mcp.tool
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b


@mcp.tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b


client = Client(mcp)

<!-- [A-a] -->
## A — What `list_tools()` actually returns

Everything downstream is built from this. One call, before any query has been asked.

In [2]:
# [A.1a]
async def bite_a1():
    async with client:
        tools = await client.list_tools()
    print(type(tools), len(tools))
    print(type(tools[0]))
    print(tools[0].name, "|", tools[0].description)
    print(tools[0].input_schema)  # <- becomes args_schema later
    print(type(tools[0].input_schema))  # <- plain dict, not a class


await bite_a1()

<class 'list'> 2
<class 'mcp_types._types.Tool'>
add | Add two integers.
{'type': 'object', 'additionalProperties': False, 'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b']}
<class 'dict'>


<!-- [B-a] -->
## B — Closures: why `wrap(mcp_tool)` takes a parameter

`wrap` and `call` in the real function only capture a variable if their own code names it —
physical nesting alone doesn't count. `mcp_tool` gets named, so it's captured correctly per
tool. Compare the two versions below.

In [3]:
# [B.1a]
# BROKEN: call() reaches directly for the loop variable `label` — a plain global lookup here,
# since this loop runs outside any function. All three closures share the one final value.
callbacks_broken = []
for label in ["a", "b", "c"]:
    def make_broken():
        def call():
            return label
        return call
    callbacks_broken.append(make_broken())

# FIXED: call() closes over a parameter captured fresh on each make_fixed() call — this is the
# wrap(mcp_tool) / call(**kwargs) pattern the real bridge uses.
callbacks_fixed = []
for label in ["a", "b", "c"]:
    def make_fixed(label):
        def call():
            return label
        return call
    callbacks_fixed.append(make_fixed(label))

print("broken:", [f() for f in callbacks_broken])
print("fixed:", [f() for f in callbacks_fixed])

broken: ['c', 'c', 'c']
fixed: ['a', 'b', 'c']


<!-- [C-a] -->
## C — Two separate wire trips, one lucky format match

`args_schema` doesn't travel to the MCP server — it travels to OpenAI, inside `create_agent`'s
own HTTP call, so the LLM knows what arguments each tool takes. It just so happens that MCP's
`list_tools()` already hands back JSON Schema, which is also exactly what OpenAI's tool-calling
API wants — so nothing needs converting. Verified below, not assumed: a `StructuredTool` built
from a raw dict keeps that dict, untouched.

In [4]:
# [C.1a]
async def call_add(**kwargs):
    async with client:
        result = await client.call_tool("add", kwargs)
    return result.data


async def bite_c1():
    async with client:
        tools = await client.list_tools()
    add_info = next(t for t in tools if t.name == "add")

    structured = StructuredTool.from_function(
        coroutine=call_add,
        name=add_info.name,
        description=add_info.description or "",
        args_schema=add_info.input_schema,  # <- raw dict going in
    )
    print("passed in: ", type(add_info.input_schema))
    print("came out as:", type(structured.args_schema))
    print(structured.args_schema)


await bite_c1()

passed in:  <class 'dict'>
came out as: <class 'dict'>
{'type': 'object', 'additionalProperties': False, 'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b']}


<!-- [D-a] -->
## D — Why re-entering `client` doesn't error

`fastmcp`'s `Client` isn't a strict open/closed file handle — its own docstring calls it
"the reentrant context manager pattern" (`fastmcp/client/client.py:958`). It keeps a
`nesting_counter`: entering increments it and reuses a live session if one exists, or starts a
fresh one if not; exiting decrements it and only tears the connection down at zero. That's why
`call` can freely re-enter a `client` that a different `async with` block already used and
closed.

In [5]:
# [D.1a]
async def bite_d1():
    async with client:
        print("first entry ok")
    async with client:
        print("second entry ok too")


await bite_d1()

first entry ok
second entry ok too


<!-- [E-a] -->
## E — Putting it together

The real function, unmodified — every piece above is one moving part inside it. The coroutine
(`call`) only actually runs once OpenAI's response is a tool-call request rather than a final
answer; see `01-architecture-overview.ipynb`, section C, for where that fits in the full round
trip.

In [6]:
# [E.1a]
async def mcp_tools_as_langchain_tools(client: Client) -> list[StructuredTool]:
    async with client:
        mcp_tools = await client.list_tools()

    def wrap(mcp_tool):
        async def call(**kwargs):
            async with client:
                result = await client.call_tool(mcp_tool.name, kwargs)
            return result.data

        return StructuredTool.from_function(
            coroutine=call,
            name=mcp_tool.name,
            description=mcp_tool.description or "",
            args_schema=mcp_tool.input_schema,
        )

    return [wrap(t) for t in mcp_tools]


async def bite_e1():
    bridged = await mcp_tools_as_langchain_tools(client)
    print([t.name for t in bridged])

    multiply_tool = next(t for t in bridged if t.name == "multiply")
    result = await multiply_tool.coroutine(a=6, b=7)  # direct call, no agent involved
    print("multiply(6, 7) via bridged tool =", result)


await bite_e1()

['add', 'multiply']


multiply(6, 7) via bridged tool = 42


<!-- [close-a] -->
That closes the loop: raw MCP `Tool` -> closure-captured `wrap`/`call` -> schema passed through
unconverted -> a reentrant client -> one real call -> a result. The same mechanism serves
`dialectica`'s actual retrieval tools in `mcp_tools.py`, just with four tools instead of two
and a real PDF behind them instead of arithmetic.